# Lab 11 — Generator-critic from scratch

Extend Lab 10's supervisor with a **critic worker** that reviews the writer's
draft against the brief, returning structured `{status, issues}`. Wire it
into a bounded refinement loop: supervisor → writer → critic → if-approve-
finalize-else-refine. Hard cap of 3 cycles.

No new dependencies. No frameworks. One new worker on top of Lab 10's
machinery.

> ⏱ Run time: 110-140 min including reading.
> 📖 The mechanism is the lab. The *why* lives in
> [`concepts/multi-agent/agent-debate-and-critics.md`](../../concepts/multi-agent/agent-debate-and-critics.md)
> and [`concepts/multi-agent/generator-critic-pattern.md`](../../concepts/multi-agent/generator-critic-pattern.md).
> Read those first — the lab references their failure modes and design rules
> directly.

## Step 0: Setup

Same setup as Lab 10. The provider-agnostic `chat_with_tools` is the
same one from Labs 01-10.

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import warnings
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


**Sample output:**

```
Using openai / gpt-4o-mini
```

## Step 1: Recap of Lab 10's machinery

Lab 10 gives us:

- `chat_with_tools(messages, tools)` — provider-agnostic LLM client with structured tool-call returns.
- `web_search(query, recency, max_results)`, `fetch_page(url, max_chars)` — Lab 03's web tools.
- `researcher_agent(question)` — Lab 03-style loop; returns `{status, findings, citations, steps}`.
- `writer_agent(brief)` — prompt-only worker; returns `{status, answer}`.
- `supervisor_agent(task)` — orchestrates researcher + writer via two worker-calling tools.
- `_action_hash(name, args)` — Lab 03 dedup; lifted to supervisor level in Lab 10.

We restate these compactly here. **Lab 11 adds one new worker and updates
the supervisor's tool registry and prompt — nothing else changes.**

### The chat client (unchanged from Lab 10)

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
    temperature: float = 0,
) -> AssistantMessage:
    """Send messages, optionally with tools; return structured response."""
    if PROVIDER == "openai":
        from openai import OpenAI

        resp = OpenAI().chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(
                    id=tc.id,
                    name=tc.function.name,
                    arguments=json.loads(tc.function.arguments),
                )
                for tc in (msg.tool_calls or [])
            ],
        )

    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {
                "name": t["function"]["name"],
                "description": t["function"]["description"],
                "input_schema": t["function"]["parameters"],
            }
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            tools=anth_tools or None,
            max_tokens=2048,
            temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [
            ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
            for b in resp.content
            if getattr(b, "type", None) == "tool_use"
        ]
        return AssistantMessage(content=text or None, tool_calls=tcs)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    """Pydantic base with extra='forbid'. Same as Lab 02."""
    model_config = ConfigDict(extra="forbid")


### The web tools and the researcher worker (unchanged from Lab 10)

For brevity we re-import these patterns here. In practice you'd refactor
Lab 10's worker functions into a small module imported by Lab 11; this
notebook re-states them inline so the lab is self-contained.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

RecencyType = Literal["any", "day", "week", "month", "year"]
_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = (
    "AgenticAIEngineer-CourseLab/0.1 "
    "(https://github.com/MHHamdan/Agentic-AI-Engineer) "
    "Mozilla/5.0 (compatible)"
)
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue", "you've reached your free article limit",
    "register to read",
]


def web_search(query: str, recency: RecencyType = "any", max_results: int = 8) -> dict:
    """Same as Lab 03 / Lab 10."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other", "detail": f"{type(e).__name__}: {e}"}
    if not raw:
        return {"status": "empty", "query": query, "detail": "no results returned"}
    return {
        "status": "ok",
        "results": [
            {"title": (r.get("title") or "").strip(),
             "url": (r.get("href") or "").strip(),
             "snippet": (r.get("body") or "").strip()}
            for r in raw if r.get("href")
        ][:max_results],
    }


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    """Same as Lab 03 / Lab 10."""
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout", "detail": "request timed out after 15s"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other", "detail": f"{type(e).__name__}: {e}"}

    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind, "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx", "detail": f"HTTP {resp.status_code}"}

    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse", "detail": f"{type(e).__name__}: {e}"}

    for tag in soup(["script", "style", "nav", "footer", "aside", "header", "form", "iframe", "noscript"]):
        tag.decompose()

    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()

    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected in body"}

    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "total_chars": len(text)}
    return {"status": "ok", "url": url, "title": title, "text": text}


# ── Researcher worker tools (Lab 10) ──
RESEARCHER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web. Returns up to max_results items with title, url, snippet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "recency": {"type": "string", "enum": ["any", "day", "week", "month", "year"]},
                    "max_results": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_page",
            "description": "Fetch the full content of a single URL.",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string"}, "max_chars": {"type": "integer"}},
                "required": ["url"],
            },
        },
    },
]


def _researcher_execute_tool(name: str, args: dict) -> dict:
    if name == "web_search":
        return web_search(args["query"], args.get("recency", "any"), args.get("max_results", 8))
    if name == "fetch_page":
        return fetch_page(args["url"], args.get("max_chars", 8000))
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


WORKER_MAX_STEPS = 8

RESEARCHER_SYSTEM_PROMPT = """You are a research worker. Answer the given question
by searching the web and reading 1-2 pages. When you have grounded findings,
respond with plain text — no further tool calls — and the loop will return
your findings to the supervisor along with the structurally-tracked citations.
Strategy:
1. web_search with 3-8 specific words from the question.
2. If snippets answer the question, summarize. Otherwise fetch_page on the
   1-2 most relevant URLs.
3. Never invent citations. Only fetched pages get cited.
"""


def researcher_agent(question: str) -> dict:
    """Same as Lab 10."""
    messages: list[dict] = [
        {"role": "system", "content": RESEARCHER_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, WORKER_MAX_STEPS + 1):
        msg = chat_with_tools(messages, tools=RESEARCHER_TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)
        if not msg.tool_calls:
            return {"status": "ok", "findings": msg.content or "",
                    "citations": citations, "steps": step}
        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"You already called {tc.name} with these arguments."}
            else:
                seen_actions.add(ah)
                tool_result = _researcher_execute_tool(tc.name, tc.arguments)
                if tc.name == "fetch_page" and tool_result.get("status") in ("ok", "too_long"):
                    citations.append({"url": tool_result["url"],
                                      "title": tool_result.get("title", "")})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:4000]})

    return {"status": "step_cap",
            "findings": ("Research did not complete within the step budget. "
                         f"I fetched {len(citations)} page(s) but did not produce a final summary."),
            "citations": citations, "steps": WORKER_MAX_STEPS}


### The writer worker (unchanged from Lab 10, with one small extension)

The writer's API takes `brief: dict`. Lab 10's brief was
`{findings, citations}`. Lab 11 extends this to optionally include
`revision_issues: list[dict]` — when the supervisor invokes the writer
a second time after a critic review, the issues get injected here and
the writer's prompt addresses them.

This is the *only* change to the writer worker. The brief shape grew
backward-compatibly.

In [ ]:
WRITER_SYSTEM_PROMPT = """You are a writer worker. You receive a brief containing
findings and a list of citations. Produce ~150 words of clean prose that:

1. States the findings accurately. Do not invent claims not present in the brief.
2. Preserves the citations. Reference them inline as [1], [2], etc. matching
   the citation list order. Then list the citations at the end as:

       [1] Title — URL
       [2] Title — URL

3. If the brief is insufficient, honestly say so. Do not paper over gaps.

If the brief includes `revision_issues`, this is a refinement round. Each issue
has a `kind` and a `detail`. Address each one explicitly in your revision.
Explain in 1 sentence what you changed.
"""


def writer_agent(brief: dict) -> dict:
    """Same shape as Lab 10's writer, extended to handle revision rounds."""
    if not brief or not brief.get("findings"):
        return {"status": "needs_more_research", "missing": ["empty or malformed brief"]}

    citations_block = "\n".join(
        f"[{i + 1}] {c.get('title', '')} — {c['url']}"
        for i, c in enumerate(brief.get("citations", []))
    ) or "(no citations)"

    issues_block = ""
    if brief.get("revision_issues"):
        issues_lines = []
        for i, issue in enumerate(brief["revision_issues"], 1):
            issues_lines.append(f"{i}. [{issue.get('kind', '?')}] {issue.get('detail', '')}")
        issues_block = (
            "\n\nREVISION REQUESTED — address each of these issues from the critic:\n"
            + "\n".join(issues_lines)
            + "\n\nEach issue must be addressed. Explain in 1 sentence what you changed."
        )

    user_prompt = (
        f"FINDINGS:\n{brief['findings']}\n\n"
        f"CITATIONS:\n{citations_block}"
        f"{issues_block}\n\n"
        f"Write the ~150-word answer following the rules in your system prompt."
    )

    msg = chat_with_tools([
        {"role": "system", "content": WRITER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])
    return {"status": "ok", "answer": msg.content or ""}


## Step 2: The critic worker

The new piece. The critic reviews `(current_draft, original_brief)` against a
5-item checklist and returns one of:

- `{"status": "ok"}` — draft passes all checks
- `{"status": "needs_revision", "issues": [{"kind": ..., "detail": ...}, ...]}` — at most 3 issues

Issue kinds form an enumerated set. The same `Literal` discipline that Lab 02
applied to tool args; we apply it here to critique categories. Enumerating
the kinds means the writer's revision logic can branch on them cleanly.

The system prompt encodes the **four rules** from the
[generator-critic pattern concept page](../../concepts/multi-agent/generator-critic-pattern.md#critic-prompt-design--the-four-rules).

In [ ]:
CRITIC_ISSUE_KINDS = [
    "unsupported_claim",   # a factual claim can't be traced to findings
    "missing_citation",    # a claim that should cite a source doesn't
    "dropped_citation",    # a citation from findings doesn't appear in draft
    "unclear_prose",       # prose is structurally hard to follow
    "format_violation",    # explicit format rule violated (e.g., word count)
]


CRITIC_SYSTEM_PROMPT = """You are a STRICT reviewer of drafts.

Your job is to apply each of these 5 checks IN ORDER to the draft you receive:

1. unsupported_claim — for each factual claim in the draft, can you find a
   supporting sentence in the FINDINGS? If no, flag it. Quote the exact claim.

2. missing_citation — does each substantive factual claim have a citation
   reference like [1] or [2]? If a claim that should cite doesn't, flag it.

3. dropped_citation — does every citation listed in CITATIONS appear in the
   draft body? If a citation is in the list but not referenced in the prose,
   flag it.

4. unclear_prose — are there sentences that are structurally hard to follow,
   contain unresolved pronouns, or contradict themselves? Be conservative —
   flag only clearly unclear cases, not stylistic preferences.

5. format_violation — does the draft meet the explicit format rules in the
   brief (e.g., ~150 words, citation format)? If a hard rule is violated,
   flag it.

RULES:
- Default to OK on borderline cases. When uncertain, return ok. Bias toward
  approval; flag only issues you can ground in specific evidence.
- Quote evidence. Each issue's `detail` should quote the specific draft
  passage and explain why it fails the check.
- Cap at 3 issues. If you find more than 3, your top 3 only — more than 3
  is a signal of structural problems, not point fixes.
- Return ONLY valid JSON. No prose preamble. No markdown fences. Just JSON.

If the draft passes all checks, respond with EXACTLY:
   {"status": "ok"}

If any check fails, respond with:
   {"status": "needs_revision", "issues": [{"kind": "<kind>", "detail": "<quote + explanation>"}, ...]}

Kinds must be one of: unsupported_claim, missing_citation, dropped_citation,
unclear_prose, format_violation.
"""


def critic_agent(draft: str, brief: dict) -> dict:
    """Review a draft against the original brief. Stateless — no revision history."""
    findings = brief.get("findings", "")
    citations = brief.get("citations", [])
    citations_block = "\n".join(
        f"[{i + 1}] {c.get('title', '')} — {c['url']}"
        for i, c in enumerate(citations)
    ) or "(none)"

    user_prompt = (
        f"FINDINGS:\n{findings}\n\n"
        f"CITATIONS:\n{citations_block}\n\n"
        f"DRAFT TO REVIEW:\n{draft}\n\n"
        f"Apply the 5 checks. Respond with JSON only."
    )

    # temperature=0 — critic is deterministic, no creativity
    msg = chat_with_tools(
        [{"role": "system", "content": CRITIC_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    raw = (msg.content or "").strip()
    # Strip code fences if the model added them despite the prompt
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        # Defensive: if the critic fails to produce JSON, treat as ok with a parse warning
        return {"status": "ok", "_parse_warning": raw[:200]}

    # Validate the envelope shape
    if result.get("status") not in ("ok", "needs_revision"):
        return {"status": "ok", "_invalid_status": result.get("status")}
    if result["status"] == "needs_revision":
        issues = result.get("issues", []) or []
        # Filter to known kinds; truncate to 3
        valid_issues = [
            i for i in issues
            if isinstance(i, dict) and i.get("kind") in CRITIC_ISSUE_KINDS
        ][:3]
        return {"status": "needs_revision", "issues": valid_issues}
    return {"status": "ok"}


## Step 3: The sycophancy diagnostic

Before integrating the critic, run the obvious-bad-draft test. Construct a
draft with deliberate failures — unsupported claims, missing citations,
contradictions — and feed it to the critic. The critic should flag it.

If your critic returns `{"status": "ok"}` on a draft like this, your
prompt is sycophantic and the full pipeline will silently approve bad
output. This diagnostic is a routine pre-deployment check; running it
takes one LLM call and saves hours of debugging.

Lab 11 ships a critic prompt designed to pass this test, so you should
see `needs_revision` with concrete issues. If you don't — investigate
before going further.

In [ ]:
# A deliberately bad draft. The findings claim X was founded in 2024
# but the draft says 1999. The draft also references a citation [3] that
# doesn't exist in the citation list.

bad_brief = {
    "findings": (
        "The Model Context Protocol (MCP) is an open standard introduced "
        "by Anthropic in 2024. It defines a client-server architecture for "
        "connecting LLM applications to external data sources and tools."
    ),
    "citations": [
        {"url": "https://anthropic.com/news/mcp", "title": "Introducing MCP"},
    ],
}

bad_draft = (
    "The Model Context Protocol (MCP) was introduced by OpenAI in 1999 [1]. "
    "It is a peer-to-peer protocol where every agent acts as a server [3]. "
    "MCP supports HTTP, WebSocket, and a custom binary protocol over UDP. "
    "The Eiffel Tower has nothing to do with this but is also relevant. "
    "MCP. MCP. MCP. MCP MCP MCP MCP. "
    "\n\n[1] Introducing MCP — https://anthropic.com/news/mcp"
)

critic_result = critic_agent(bad_draft, bad_brief)
print(json.dumps(critic_result, indent=2))


**Sample output (your exact issues will vary; the key check is that `status != "ok"`):**

```json
{
  "status": "needs_revision",
  "issues": [
    {
      "kind": "unsupported_claim",
      "detail": "Claim 'introduced by OpenAI in 1999' contradicts the findings, which state MCP was introduced by Anthropic in 2024."
    },
    {
      "kind": "missing_citation",
      "detail": "Citation [3] is referenced in the draft but does not exist in the citations list."
    },
    {
      "kind": "unclear_prose",
      "detail": "The repetition 'MCP. MCP. MCP.' and the Eiffel Tower reference are off-topic and structurally unclear."
    }
  ]
}
```

The critic correctly flags the unsupported claim (year and organization wrong),
the missing citation ([3] doesn't exist), and the unclear prose. If your run
returns `{"status": "ok"}` here, your critic prompt is sycophantic — re-read
the four rules in the concept page and check your prompt.

## Step 4: Wire the critic into the supervisor

Three updates to Lab 10's supervisor:

1. Add `call_critic` to `SUPERVISOR_TOOLS` with a `CallCriticArgs` schema.
2. Update `SUPERVISOR_SYSTEM_PROMPT` to describe the refinement loop.
3. Raise `SUPERVISOR_MAX_STEPS` from 6 to 10 to accommodate refinement cycles.
4. Add `MAX_REFINEMENT_CYCLES = 3` as a hard cap the supervisor enforces.

The supervisor still uses the standard `chat_with_tools` dispatch. Lab 11's
"refinement loop" is not a new control structure — it's an emergent behavior
of the supervisor's agent loop calling `call_writer` and `call_critic` in
sequence, guided by the system prompt and bounded by the hard cap.

In [ ]:
SUPERVISOR_MAX_STEPS = 10        # raised from 6 in Lab 10
MAX_REFINEMENT_CYCLES = 3        # hard cap; supervisor enforces


# ── Worker tool schemas ─────────────────────────────────────────────────

class CallResearcherArgs(StrictModel):
    question: str = Field(
        description="The question for the researcher. Self-contained, clear."
    )


class CallWriterArgs(StrictModel):
    findings: str = Field(
        description="Research findings to turn into prose."
    )
    citations: list[dict] = Field(
        default_factory=list,
        description="Citation objects [{url, title}, ...] from the researcher.",
    )
    revision_issues: list[dict] = Field(
        default_factory=list,
        description=(
            "Critic-flagged issues from the previous round. Pass EXACTLY as "
            "received from call_critic; do not paraphrase. Empty list on the "
            "first call to the writer."
        ),
    )


class CallCriticArgs(StrictModel):
    draft: str = Field(
        description="The current draft to review. Pass the writer's output verbatim."
    )
    findings: str = Field(
        description="The ORIGINAL research findings (not the revised brief)."
    )
    citations: list[dict] = Field(
        default_factory=list,
        description="The ORIGINAL citations from the researcher.",
    )


def _call_researcher_tool(args: CallResearcherArgs) -> dict:
    return researcher_agent(args.question)


def _call_writer_tool(args: CallWriterArgs) -> dict:
    return writer_agent({
        "findings": args.findings,
        "citations": args.citations,
        "revision_issues": args.revision_issues,
    })


def _call_critic_tool(args: CallCriticArgs) -> dict:
    return critic_agent(
        draft=args.draft,
        brief={"findings": args.findings, "citations": args.citations},
    )


SUPERVISOR_TOOLS = {
    "call_researcher": (
        _call_researcher_tool, CallResearcherArgs,
        "Send a question to the researcher worker. Returns {findings, citations}. "
        "Use when fresh information is needed. Do NOT use if the answer is in "
        "this conversation already. Do NOT call more than once with the same question.",
    ),
    "call_writer": (
        _call_writer_tool, CallWriterArgs,
        "Send findings + citations + (optionally) revision_issues to the writer. "
        "Returns {status, answer}. Call once initially with empty revision_issues. "
        "On subsequent calls during a refinement cycle, pass the critic's issues. "
        "Do NOT skip this — it's how the supervisor produces the final prose.",
    ),
    "call_critic": (
        _call_critic_tool, CallCriticArgs,
        "Review a draft against the original brief. Returns {status, issues}. "
        "Call this after call_writer. If status='needs_revision', call call_writer "
        "again with the issues. Repeat until status='ok' or you have reached the "
        "MAX_REFINEMENT_CYCLES = 3 cap. Always pass the ORIGINAL findings/citations "
        "to the critic, never an accumulated/revised brief.",
    ),
}


def _supervisor_make_schemas() -> list[dict]:
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": args_model.model_json_schema(),
            },
        }
        for name, (_fn, args_model, description) in SUPERVISOR_TOOLS.items()
    ]


def _supervisor_execute_tool(call: ToolCall) -> dict:
    if call.name not in SUPERVISOR_TOOLS:
        return {"status": "error", "kind": "unknown_worker",
                "tool": call.name, "available": list(SUPERVISOR_TOOLS)}
    fn, args_model, _ = SUPERVISOR_TOOLS[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"status": "error", "kind": "supervisor_dispatch_error",
                "detail": f"{type(e).__name__}: {e}"}


SUPERVISOR_SYSTEM_PROMPT = """You are a supervisor agent coordinating three workers:
a researcher (web tools), a writer (prose), and a critic (reviewer).

Your job:
1. Call call_researcher to get findings + citations.
2. Call call_writer with the findings + empty revision_issues to get an initial draft.
3. Call call_critic with the draft + original findings/citations.
4. If the critic returns status='ok', return the writer's draft as your final answer.
5. If the critic returns status='needs_revision', call call_writer AGAIN with the
   same findings/citations BUT with the critic's issues passed as revision_issues.
   Then call_critic on the new draft.
6. Repeat step 5 up to MAX_REFINEMENT_CYCLES = 3 times. After the 3rd refinement
   cycle, finalize whatever the writer last produced — even if the critic still
   wants changes. Mention the unresolved issues honestly in your final answer.

CRITICAL:
- Citations from the researcher MUST be preserved in the final answer.
- Always pass the ORIGINAL findings/citations to call_critic, never the revised brief.
- Do not loop. The action-hash system will refuse repeated calls; if it does,
  move on instead of trying the same thing again.
- Do not call the same worker more than necessary.

Track refinement cycles in your reasoning. A "cycle" is one (writer + critic)
round. The first writer call (with empty revision_issues) is cycle 0; if the
critic flags it, then writer-again is cycle 1; and so on.
"""


def supervisor_agent(task: str, verbose: bool = True) -> dict:
    """Run the supervisor with researcher + writer + critic + refinement loop."""
    messages: list[dict] = [
        {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    seen_actions: set[str] = set()
    trace: list[dict] = []
    schemas = _supervisor_make_schemas()
    refinement_cycles = 0

    for step in range(1, SUPERVISOR_MAX_STEPS + 1):
        if verbose:
            print(f"\n── Supervisor step {step} ──")

        msg = chat_with_tools(messages, tools=schemas)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            if verbose:
                print(f"  ◆ FINAL: {(msg.content or '')[:120]}...")
            return {"answer": msg.content or "", "trace": trace, "steps": step,
                    "refinement_cycles": refinement_cycles,
                    "stopped_reason": "supervisor_returned"}

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"You already called {tc.name} with these args."}
                if verbose:
                    print(f"  ✗ {tc.name}(...) [REPEATED]")
            else:
                seen_actions.add(ah)
                if verbose:
                    print(f"  → {tc.name}(...)")

                # Enforce the refinement cap. Count writer calls AFTER the first.
                if tc.name == "call_writer":
                    is_revision = bool(tc.arguments.get("revision_issues"))
                    if is_revision:
                        refinement_cycles += 1
                        if refinement_cycles > MAX_REFINEMENT_CYCLES:
                            tool_result = {
                                "status": "refinement_cap_reached",
                                "detail": (f"Refinement cap of {MAX_REFINEMENT_CYCLES} "
                                           "cycles reached. Finalize the last draft and "
                                           "surface any remaining critic issues honestly."),
                            }
                            messages.append({"role": "tool", "tool_call_id": tc.id,
                                             "content": json.dumps(tool_result)})
                            trace.append({"step": step, "tool": tc.name,
                                          "result_status": "refinement_cap_reached"})
                            continue

                tool_result = _supervisor_execute_tool(tc)
                if verbose:
                    summary = (tool_result.get("answer") or
                               tool_result.get("findings") or
                               json.dumps(tool_result)[:80])
                    summary = str(summary)[:80]
                    print(f"  ← status={tool_result.get('status', '?')}: {summary}...")

            trace.append({"step": step, "tool": tc.name,
                          "result_status": tool_result.get("status")})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:6000]})

    return {"answer": "[supervisor hit step cap]", "trace": trace,
            "steps": SUPERVISOR_MAX_STEPS, "refinement_cycles": refinement_cycles,
            "stopped_reason": "step_cap"}


## Step 5: Run the supervisor end-to-end

The full pipeline on a real task. Expect roughly:

- 1 researcher call (returns findings + citations).
- 1 writer call (initial draft, empty `revision_issues`).
- 1 critic call (reviews the draft).
- 0-2 refinement cycles, each a writer + critic pair (capped at 3).
- 1 supervisor finalization call.

Total typical: 5-9 supervisor steps; trace length 4-8 tool calls. Cost
roughly 2x Lab 10 in the common case.

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "and write a 150-word summary."
)
result = supervisor_agent(task)

print("\n" + "=" * 70)
print("FINAL ANSWER:")
print("=" * 70)
print(result["answer"])
print()
print(f"Steps: {result['steps']}, refinement cycles: {result['refinement_cycles']}, "
      f"stopped: {result['stopped_reason']}")
print(f"Tool calls in trace: {len(result['trace'])}")
print("\nTrace summary:")
for t in result["trace"]:
    print(f"  step {t['step']}: {t['tool']:<16s} → {t['result_status']}")


**Sample output (LLM responses will vary; trajectory should be stable):**

```
── Supervisor step 1 ──
  → call_researcher(...)
  ← status=ok: MCP (Model Context Protocol) is an open standard introduced by ...

── Supervisor step 2 ──
  → call_writer(...)
  ← status=ok: The Model Context Protocol (MCP) is an open standard introduced...

── Supervisor step 3 ──
  → call_critic(...)
  ← status=needs_revision: {"status": "needs_revision", "issues": [{"kind"...

── Supervisor step 4 ──
  → call_writer(...)         ← refinement cycle 1
  ← status=ok: The Model Context Protocol (MCP), introduced by Anthropic in...

── Supervisor step 5 ──
  → call_critic(...)
  ← status=ok: {"status": "ok"}...

── Supervisor step 6 ──
  ◆ FINAL: The Model Context Protocol (MCP), introduced by Anthropic in 2024...

======================================================================
FINAL ANSWER:
======================================================================
The Model Context Protocol (MCP) [1], introduced by Anthropic in late 2024,
is an open standard for connecting LLM applications to external data sources
and tools. It defines a client-server architecture in which MCP servers
expose resources, tools, and prompts to MCP clients (typically LLM apps)...

[1] Introducing MCP — https://anthropic.com/news/mcp
[2] MCP Specification — https://modelcontextprotocol.io/docs

Steps: 6, refinement cycles: 1, stopped: supervisor_returned
Tool calls in trace: 5

Trace summary:
  step 1: call_researcher  → ok
  step 2: call_writer      → ok
  step 3: call_critic      → needs_revision
  step 4: call_writer      → ok
  step 5: call_critic      → ok
```

One refinement cycle in this run — the critic flagged an issue in draft v1,
the writer addressed it in v2, the critic approved v2. The trajectory is
linear: researcher → writer → critic → (refine) → writer → critic → finalize.

## Step 6: Failure-mode walkthrough

The four debate-specific failure modes, and the mitigations Lab 11
ships against each.

### Failure mode 1: Sycophancy

**Symptom:** Every critic call returns `{"status": "ok"}` regardless of draft
quality. The refinement loop never fires.

**Diagnostic:** Run Step 3's obvious-bad-draft test. If the critic doesn't flag
it, the prompt is sycophantic.

**Mitigations Lab 11 uses:**

- Rubric-anchored prompt: 5 specific checks the critic applies in order, with
  evidence-grounded issues. Replaces "review this draft" (vibes) with concrete
  pass/fail decisions.
- Strict-reviewer framing: explicit "you are a STRICT reviewer; default to ok
  only when the draft genuinely meets the rubric."
- `temperature=0` for the critic: deterministic, no creativity-induced agreement.

**What Lab 11 does NOT use:**

- Different model for the critic. Single-provider keeps the lab simple but is
  the strongest single mitigation if your budget allows two API costs. Try it
  as an exercise — pass `model="claude-haiku-4-5-20251001"` to the critic's
  `chat_with_tools` call while the writer uses `gpt-4o-mini`.

### Failure mode 2: Infinite agreement

**Symptom:** Critic approves; draft is mediocre; nobody notices because both
agents are happy.

**Diagnostic:** Eval-time only — you can't catch this in real-time. Spot-check
samples manually or with the Lab 09 eval harness applied to the final outputs.

**Mitigation Lab 11 uses:** the same as sycophancy. Rubric anchoring is the
single biggest defense. The diagnostic from Step 3 isn't a one-time check —
re-run it on samples of your task's actual brief shape.

**What Lab 11 does NOT solve:** the underlying ceiling of the model. If both
the generator and critic are at the model's quality ceiling, no amount of
critique will push past it. You need better data, better tools, or a stronger
model.

### Failure mode 3: Runaway disagreement

**Symptom:** The critic always finds new issues; refinement cycles never
converge.

**Diagnostic:** The `refinement_cycles` field in the supervisor's return
hitting the cap on tasks that should converge.

In [ ]:
# Demonstrate the cap mechanism with a synthetic trace
print("Hard cap mechanism:")
print(f"  MAX_REFINEMENT_CYCLES = {MAX_REFINEMENT_CYCLES}")
print(f"  SUPERVISOR_MAX_STEPS  = {SUPERVISOR_MAX_STEPS}")
print()
print("If the supervisor tries to call call_writer with revision_issues")
print(f"for the {MAX_REFINEMENT_CYCLES + 1}th time, the supervisor's loop returns:")
print(json.dumps({
    "status": "refinement_cap_reached",
    "detail": (f"Refinement cap of {MAX_REFINEMENT_CYCLES} cycles reached. "
               "Finalize the last draft and surface any remaining critic "
               "issues honestly."),
}, indent=2))
print()
print("The supervisor's system prompt instructs it to honor this and surface")
print("the unresolved critic issues in the final answer — not to silently approve.")


**Mitigation Lab 11 uses:** the `MAX_REFINEMENT_CYCLES = 3` cap, enforced
inside the supervisor's loop. Critically, the system prompt tells the
supervisor what to *do* when the cap fires — finalize with honest surfacing —
rather than trying to "fix" the runaway.

**What Lab 11 explicitly does NOT do:** allow the supervisor to keep retrying.
A wandering supervisor terminating sooner is better than a wandering supervisor
trying harder.

### Failure mode 4: Critique drift

**Symptom:** Critic flags X in round 1; flags Y while accepting X in round 2;
flags Z while accepting X and Y in round 3. Standards shift across rounds.

**Mitigation Lab 11 uses:** the critic is **stateless**. It receives
`(current_draft, original_brief)` only — never the revision history. Each
critique is a fresh judgment against the same original rubric. This is enforced
by the `CallCriticArgs` schema, which takes `findings` + `citations` but no
revision context.

**Why this matters:** if you "give the critic context" (a list of past critic
results, or an accumulated brief), you make the critic's judgments
path-dependent, which is the *opposite* of what you want. Resist the urge.

## Step 7 (stretch): The self-critique variation

Lab 11 uses **separate-critic** by default — distinct generator and critic
agents. The alternative is **self-critique**: same agent generates *and*
reviews.

Self-critique:

- ✅ Cheaper (one fewer agent prompt setup)
- ✅ Faster (one fewer round-trip)
- ❌ More sycophantic (same context, same blind spots)
- ❌ Catches obvious errors but misses subtle ones at a higher rate

The cost-quality trade is real. For high-stakes tasks, separate-critic is
the right baseline. For speed-critical tasks where you'd rather have *some*
review than none, self-critique is a defensible compromise.

Below, a sketch of self-critique applied to the writer. Run it on the
bad-draft diagnostic from Step 3 and compare to the separate critic's output.

In [ ]:
SELF_CRITIQUE_PROMPT = """You produced this draft. Now review it strictly
against the brief. Apply the same 5 checks the critic uses:
unsupported_claim, missing_citation, dropped_citation, unclear_prose,
format_violation.

Return JSON only. Status 'ok' if all pass; 'needs_revision' with up to 3
specific issues otherwise. Default to ok on borderline cases.

BRIEF FINDINGS: {findings}
BRIEF CITATIONS: {citations}
DRAFT YOU PRODUCED: {draft}
"""


def self_critique(draft: str, brief: dict) -> dict:
    """The writer reviews its own draft. Cheaper, more sycophantic."""
    citations_block = "\n".join(
        f"[{i + 1}] {c.get('title', '')} — {c['url']}"
        for i, c in enumerate(brief.get("citations", []))
    ) or "(none)"
    prompt = SELF_CRITIQUE_PROMPT.format(
        findings=brief.get("findings", ""),
        citations=citations_block,
        draft=draft,
    )
    msg = chat_with_tools(
        # Note: NO separate "strict reviewer" system prompt. Same WRITER prompt context.
        [{"role": "system", "content": WRITER_SYSTEM_PROMPT},
         {"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = (msg.content or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"status": "ok", "_parse_warning": raw[:200]}


# Comparison: separate-critic vs self-critique on the bad draft
print("─── Separate critic on the bad draft ───")
print(json.dumps(critic_agent(bad_draft, bad_brief), indent=2))
print("\n─── Self-critique on the same bad draft ───")
print(json.dumps(self_critique(bad_draft, bad_brief), indent=2))


**Sample output (results vary; pattern is stable):**

In typical runs the separate critic flags 2-3 specific issues while
self-critique either misses some issues, returns weaker/more-hedged issue
descriptions, or returns `ok` outright. The empirical gap is real and
reproducible — it's the reason production critique systems use separate-critic
as the default.

Reasonable rules of thumb:

- Use **separate-critic** when factual correctness matters.
- Use **self-critique** when you need *some* review on a tight latency budget.
- Use **both** — self-critique as a cheap first pass, separate-critic only on
  drafts that pass self-critique — as a cost-effective compromise for medium-
  stakes tasks.

## What you just built

A 4-agent supervisor-worker system with a bounded refinement loop, in roughly
350 lines of Python total (Lab 10's ~250 + Lab 11's ~100 delta):

- The supervisor, researcher, and writer from Lab 10 — **unchanged**.
- One new worker: the critic, with structured `{status, issues}` envelope and
  enumerated `kind` field.
- Three updates to the supervisor: one new tool (`call_critic`), an updated
  prompt describing the refinement loop, and a hard cap of 3 refinement
  cycles enforced inside the loop.
- The four critic-prompt-design rules (anchor to checklist, default to ok,
  require evidence, bound issue list), each preventing a specific failure
  mode.
- The obvious-bad-draft sycophancy diagnostic as a routine pre-deployment check.
- The four debate failure modes (sycophancy, infinite agreement, runaway
  disagreement, critique drift) with the mitigation each gets.

The patterns from Path 01 + Lab 10 transferred directly. The agent loop is
unchanged. Workers are still functions the supervisor calls as tools. The
critic *is* a worker — not a special concept, not a different abstraction,
just another tool in the supervisor's registry. **Iterative refinement turns
out to be a few prompt changes and one new worker, not a new framework.**

That's the bet of Path 03: build the mechanism from first principles first;
understand its failure modes; *then* adopt frameworks (CrewAI, AutoGen,
LangGraph multi-agent) as ergonomic improvements once you know what they're
hiding.

## Production readiness — out of scope here

For a real deployment you'd also want: cost-budget enforcement per refinement
cycle (the critic + writer pair averages 2x the single-pass cost; production
systems track this and can short-circuit when budget is exhausted);
critique-quality eval (the Lab 09 harness, applied to whether the critic's
flagged issues *actually were* issues in the draft); cached critique results
for identical (draft, brief) pairs; structured logging at every handoff
(worker name, timing, cycle number, critic status); circuit breakers per worker.

## Next

- Take the [agent debate and critics quiz](../../quizzes/multi-agent/agent-debate-and-critics.md).
- Path 03 continues with Module 3 (plan-and-execute) in a future batch.
- If you've also done Path 02, the future multi-agent RAG batch composes
  Lab 06-08's retrieval pipeline with the supervisor + critic patterns from
  Labs 10-11.